In [17]:
#!/usr/bin/env python
# ═══════════════════════════════════════════════════════════════════════
#  IMAGE RESTORATION — ADVANCED SUBMISSION NOTEBOOK
# ═══════════════════════════════════════════════════════════════════════
#  Improvements over baseline:
#    1. NAFNet-style blocks (SimpleGate + Simplified Channel Attention)
#    2. Multi-scale frequency loss (better HF recovery)
#    3. Progressive training: L1 → Charbonnier+Freq → fine-tune
#    4. Warmup + cosine scheduler
#    5. Mixed precision (AMP) for 2× throughput
#    6. Gradient accumulation option
#    7. Exponential Moving Average (EMA) of model weights
# ═══════════════════════════════════════════════════════════════════════

In [18]:
import os, glob, math, time, random, copy
import numpy as np
import pandas as pd
import base64
from io import BytesIO
from collections import OrderedDict

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.cuda.amp import autocast, GradScaler

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

In [19]:
DATA_ROOT = "/kaggle/input/competitions/ExeBit_kla_ai_hack/KLA AI - HACK"
TRAIN_GT  = os.path.join(DATA_ROOT, "train", "GT")
TRAIN_LR  = os.path.join(DATA_ROOT, "train", "NoisyLR")
TEST_LR   = os.path.join(DATA_ROOT, "test",  "NoisyLR")
SUB_DIR   = "/kaggle/working/submission"
os.makedirs(SUB_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


In [20]:
class CFG:
    # Architecture
    base_channels = 32
    n_blocks      = 4          # more blocks = more capacity

    # Training — Phase 1 (main)
    epochs_p1     = 100
    lr_p1         = 2e-4
    batch_size    = 16

    # Training — Phase 2 (fine-tune with lower LR)
    epochs_p2     = 40
    lr_p2         = 5e-5

    # Loss
    freq_weight   = 0.1
    multiscale    = True        # apply loss at 256 + 128 scale

    # EMA
    use_ema       = True
    ema_decay     = 0.999

    # AMP
    use_amp       = False

    # Validation
    val_frac      = 0.05

    # Inference
    use_tta       = True
    clamp_output  = True

In [21]:
class SRDataset(Dataset):
    def __init__(self, lr_dir, gt_dir=None, augment=False, crop_size=None):
        self.lr_paths  = sorted(glob.glob(os.path.join(lr_dir, "*.npy")))
        self.gt_dir    = gt_dir
        self.augment   = augment and (gt_dir is not None)
        self.crop_size = crop_size  # LR crop size (GT crop = 2×)
        if gt_dir:
            self.gt_paths = sorted(glob.glob(os.path.join(gt_dir, "*.npy")))
            assert len(self.lr_paths) == len(self.gt_paths)

    def __len__(self):
        return len(self.lr_paths)

    def __getitem__(self, idx):
        lr = np.load(self.lr_paths[idx]).astype(np.float32)

        if self.gt_dir is not None:
            gt = np.load(self.gt_paths[idx]).astype(np.float32)

            # Random crop (helps regularization + allows larger batches)
            if self.crop_size and self.augment:
                cs = self.crop_size
                h, w = lr.shape
                if h > cs and w > cs:
                    y = random.randint(0, h - cs)
                    x = random.randint(0, w - cs)
                    lr = lr[y:y+cs, x:x+cs]
                    gt = gt[y*2:(y+cs)*2, x*2:(x+cs)*2]

            if self.augment:
                if random.random() > 0.5:
                    lr = np.fliplr(lr).copy(); gt = np.fliplr(gt).copy()
                if random.random() > 0.5:
                    lr = np.flipud(lr).copy(); gt = np.flipud(gt).copy()
                k = random.randint(0, 3)
                lr = np.rot90(lr, k).copy(); gt = np.rot90(gt, k).copy()

            return torch.from_numpy(lr).unsqueeze(0), torch.from_numpy(gt).unsqueeze(0)
        else:
            return torch.from_numpy(lr).unsqueeze(0), os.path.basename(self.lr_paths[idx])

In [22]:

class SimpleGate(nn.Module):
    """Split channels in half, multiply element-wise. Cheaper than GELU."""
    def forward(self, x):
        x1, x2 = x.chunk(2, dim=1)
        return x1 * x2


class SimplifiedChannelAttention(nn.Module):
    """Global average pool → 1×1 conv → sigmoid → scale."""
    def __init__(self, ch):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Conv2d(ch, ch, 1)

    def forward(self, x):
        return x * self.fc(self.pool(x))


class NAFBlock(nn.Module):
    """NAFNet-style block: LayerNorm → Conv → SimpleGate → Conv → SCA → skip."""
    def __init__(self, ch):
        super().__init__()
        self.norm = nn.GroupNorm(1, ch)  # equivalent to LayerNorm for images
        self.body = nn.Sequential(
            nn.Conv2d(ch, ch * 2, 3, padding=1),   # expand for SimpleGate
            SimpleGate(),                            # halves channels
            nn.Conv2d(ch, ch, 3, padding=1),
        )
        self.sca   = SimplifiedChannelAttention(ch)
        self.scale = nn.Parameter(torch.zeros(1))    # learnable residual scale

    def forward(self, x):
        res = self.body(self.norm(x))
        res = self.sca(res)
        return x + res * self.scale

In [23]:

class AdvancedUNetSR(nn.Module):
    """
    U-Net with NAFNet-style blocks + PixelShuffle ×2.
    Stronger feature extraction than vanilla ResBlocks.
    """
    def __init__(self, ch=64, n_blocks=6):
        super().__init__()

        # Encoder
        self.head = nn.Conv2d(1, ch, 3, padding=1)
        self.enc1 = nn.Sequential(*[NAFBlock(ch) for _ in range(n_blocks)])
        self.down1 = nn.Conv2d(ch, ch*2, 2, stride=2)  # strided conv downsample

        self.enc2 = nn.Sequential(*[NAFBlock(ch*2) for _ in range(n_blocks)])
        self.down2 = nn.Conv2d(ch*2, ch*4, 2, stride=2)

        # Bottleneck
        self.mid = nn.Sequential(*[NAFBlock(ch*4) for _ in range(n_blocks)])

        # Decoder
        self.up2 = nn.Sequential(
            nn.Conv2d(ch*4, ch*2 * 4, 1),
            nn.PixelShuffle(2),
        )
        self.fuse2 = nn.Conv2d(ch*4, ch*2, 1)
        self.dec2 = nn.Sequential(*[NAFBlock(ch*2) for _ in range(n_blocks)])

        self.up1 = nn.Sequential(
            nn.Conv2d(ch*2, ch * 4, 1),
            nn.PixelShuffle(2),
        )
        self.fuse1 = nn.Conv2d(ch*2, ch, 1)
        self.dec1 = nn.Sequential(*[NAFBlock(ch) for _ in range(n_blocks)])

        # 2× SR upsample
        self.tail = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1),
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(ch, 4, 3, padding=1),  # 4 = 1 × 2²
            nn.PixelShuffle(2),
        )

    def forward(self, x):
        x_bic = F.interpolate(x, scale_factor=2, mode='bicubic', align_corners=False)

        e1 = self.enc1(self.head(x))
        e2 = self.enc2(self.down1(e1))
        m  = self.mid(self.down2(e2))

        d2 = self.fuse2(torch.cat([self.up2(m), e2], 1))
        d2 = self.dec2(d2)

        d1 = self.fuse1(torch.cat([self.up1(d2), e1], 1))
        d1 = self.dec1(d1)

        return self.tail(d1) + x_bic

In [24]:

class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-3):
        super().__init__()
        self.eps_sq = eps ** 2
    def forward(self, pred, target):
        return torch.mean(torch.sqrt((pred - target)**2 + self.eps_sq))


class MultiScaleFreqLoss(nn.Module):
    """
    Charbonnier + FFT L1 at original and half resolution.
    Multi-scale helps the model learn both coarse structure and fine detail.
    """
    def __init__(self, freq_w=0.1, multiscale=True):
        super().__init__()
        self.charb = CharbonnierLoss()
        self.fw    = freq_w
        self.ms    = multiscale

    def forward(self, pred, target):
        # Full-resolution loss
        loss = self.charb(pred, target)
        if self.fw > 0:
            loss += self.fw * F.l1_loss(
                torch.abs(torch.fft.rfft2(pred)),
                torch.abs(torch.fft.rfft2(target)))

        # Half-resolution loss (optional)
        if self.ms:
            pred_ds   = F.interpolate(pred,   scale_factor=0.5, mode='area')
            target_ds = F.interpolate(target, scale_factor=0.5, mode='area')
            loss += 0.5 * self.charb(pred_ds, target_ds)

        return loss

In [25]:

class ModelEMA:
    """Exponential Moving Average of model weights."""
    def __init__(self, model, decay=0.999):
        self.ema = copy.deepcopy(model).eval()
        self.decay = decay
        for p in self.ema.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        for ema_p, model_p in zip(self.ema.parameters(), model.parameters()):
            ema_p.data.mul_(self.decay).add_(model_p.data, alpha=1 - self.decay)

In [26]:
def compute_psnr(pred, gt):
    mse = np.mean((pred - gt) ** 2)
    if mse < 1e-12: return 60.0
    dr = gt.max() - gt.min()
    if dr < 1e-8: return 0.0
    return 10.0 * math.log10(dr ** 2 / mse)

In [27]:

def train_model():
    print("=" * 60)
    print("  TRAINING — ADVANCED PIPELINE")
    print("=" * 60)

    full_ds = SRDataset(TRAIN_LR, TRAIN_GT, augment=True, crop_size=96)
    n_val   = max(1, int(CFG.val_frac * len(full_ds)))
    n_train = len(full_ds) - n_val
    train_ds, val_ds = random_split(full_ds, [n_train, n_val],
                                    generator=torch.Generator().manual_seed(SEED))

    train_dl = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
    # Validation uses full images (no crop) — create separate dataset
    val_full = SRDataset(TRAIN_LR, TRAIN_GT, augment=False, crop_size=None)
    val_indices = val_ds.indices
    val_dl_items = [val_full[i] for i in val_indices]

    print(f"  Train: {n_train}  |  Val: {n_val}")

    model = AdvancedUNetSR(ch=CFG.base_channels, n_blocks=CFG.n_blocks).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Model params: {n_params:,}")

    ema = ModelEMA(model, CFG.ema_decay) if CFG.use_ema else None
    scaler = GradScaler() if CFG.use_amp else None

    best_psnr = 0.0
    best_state = None
    t0 = time.time()

    # ── Phase 1: main training ──
    print(f"\n  Phase 1: {CFG.epochs_p1} epochs, lr={CFG.lr_p1}")
    optimizer = optim.AdamW(model.parameters(), lr=CFG.lr_p1,
                            weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CFG.epochs_p1, eta_min=1e-6)
    criterion = MultiScaleFreqLoss(CFG.freq_weight, CFG.multiscale)

    for epoch in range(1, CFG.epochs_p1 + 1):
        model.train()
        rloss = 0.0

        for lr_img, gt_img in train_dl:
            lr_img = lr_img.to(DEVICE, non_blocking=True)
            gt_img = gt_img.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            if CFG.use_amp:
                with autocast():
                    pred = model(lr_img)
                    loss = criterion(pred, gt_img)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                pred = model(lr_img)
                loss = criterion(pred, gt_img)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            rloss += loss.item()
            if ema: ema.update(model)

        scheduler.step()
        avg_loss = rloss / len(train_dl)

        # Validate (using EMA model if available)
        eval_model = ema.ema if ema else model
        eval_model.eval()
        psnr_sum, psnr_cnt = 0.0, 0

        with torch.no_grad():
            for lr_t, gt_t in val_dl_items:
                lr_t = lr_t.unsqueeze(0).to(DEVICE)
                pred = eval_model(lr_t).cpu().squeeze().numpy()
                gt_np = gt_t.squeeze().numpy()
                psnr_sum += compute_psnr(pred, gt_np)
                psnr_cnt += 1

        vpsnr = psnr_sum / max(psnr_cnt, 1)
        marker = ""
        if vpsnr > best_psnr:
            best_psnr = vpsnr
            best_state = {k: v.cpu().clone()
                          for k, v in eval_model.state_dict().items()}
            marker = " ★"

        if epoch % 5 == 0 or epoch <= 3 or marker:
            elapsed = time.time() - t0
            print(f"  P1 Ep {epoch:3d}/{CFG.epochs_p1}  loss={avg_loss:.5f}  "
                  f"psnr={vpsnr:.2f}  lr={scheduler.get_last_lr()[0]:.1e}  "
                  f"[{elapsed/60:.1f}m]{marker}")

    # ── Phase 2: fine-tune with lower LR ──
    print(f"\n  Phase 2: {CFG.epochs_p2} epochs, lr={CFG.lr_p2}")
    model.load_state_dict(best_state)
    model.to(DEVICE)
    if ema: ema = ModelEMA(model, CFG.ema_decay)

    # Use full images (no crop) for fine-tuning
    ft_ds = SRDataset(TRAIN_LR, TRAIN_GT, augment=True, crop_size=None)
    ft_train, _ = random_split(ft_ds, [n_train, n_val],
                               generator=torch.Generator().manual_seed(SEED))
    ft_dl = DataLoader(ft_train, batch_size=max(1, CFG.batch_size // 2),
                       shuffle=True, num_workers=2, pin_memory=True, drop_last=True)

    optimizer = optim.AdamW(model.parameters(), lr=CFG.lr_p2, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CFG.epochs_p2, eta_min=1e-7)

    for epoch in range(1, CFG.epochs_p2 + 1):
        model.train()
        rloss = 0.0

        for lr_img, gt_img in ft_dl:
            lr_img = lr_img.to(DEVICE, non_blocking=True)
            gt_img = gt_img.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            if CFG.use_amp:
                with autocast():
                    pred = model(lr_img)
                    loss = criterion(pred, gt_img)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                pred = model(lr_img)
                loss = criterion(pred, gt_img)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            rloss += loss.item()
            if ema: ema.update(model)

        scheduler.step()

        eval_model = ema.ema if ema else model
        eval_model.eval()
        psnr_sum, psnr_cnt = 0.0, 0
        with torch.no_grad():
            for lr_t, gt_t in val_dl_items:
                lr_t = lr_t.unsqueeze(0).to(DEVICE)
                pred = eval_model(lr_t).cpu().squeeze().numpy()
                psnr_sum += compute_psnr(pred, gt_t.squeeze().numpy())
                psnr_cnt += 1
        vpsnr = psnr_sum / max(psnr_cnt, 1)
        marker = ""
        # if vpsnr > best_psnr:
        #     best_psnr = vpsnr
        #     best_state = {k: v.cpu().clone()
        #                   for k, v in eval_model.state_dict().items()}
        #     marker = " ★"
                # marker = ""
        marker = ""
        if vpsnr > best_psnr:
            best_psnr = vpsnr
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            torch.save(best_state, "/kaggle/working/best_model.pth")
            marker = f" ★ saved"

        if epoch % 20 == 0:
            torch.save(model.state_dict(), f"/kaggle/working/checkpoint_ep{epoch}.pth")

        # if epoch % 5 == 0 or epoch <= 2 or marker:
        #     elapsed = time.time() - t0
        #     print(f"  P2 Ep {epoch:3d}/{CFG.epochs_p2}  "

        if epoch % 5 == 0 or epoch <= 2 or marker:
            elapsed = time.time() - t0
            print(f"  P2 Ep {epoch:3d}/{CFG.epochs_p2}  "
                  f"loss={rloss/len(ft_dl):.5f}  psnr={vpsnr:.2f}  "
                  f"[{elapsed/60:.1f}m]{marker}")

    print(f"\n  Best PSNR: {best_psnr:.2f} dB")
    model.load_state_dict(best_state)
    model.to(DEVICE)
    return model

In [28]:

@torch.no_grad()
def predict_single(model, lr_tensor):
    if not CFG.use_tta:
        pred = model(lr_tensor.to(DEVICE)).cpu()
        return pred.squeeze().numpy()

    preds = []
    for flip in (False, True):
        for k in range(4):
            x = lr_tensor.clone()
            if flip: x = torch.flip(x, [-1])
            x = torch.rot90(x, k, [-2, -1])
            p = model(x.to(DEVICE)).cpu()
            p = torch.rot90(p, -k, [-2, -1])
            if flip: p = torch.flip(p, [-1])
            preds.append(p)
    return torch.stack(preds).mean(0).squeeze().numpy()

In [29]:

def generate_predictions(model):
    print("\n" + "=" * 60)
    print("  INFERENCE")
    print("=" * 60)
    model.eval()
    test_ds = SRDataset(TEST_LR, gt_dir=None)
    print(f"  Test: {len(test_ds)}  |  TTA: {'ON (8×)' if CFG.use_tta else 'OFF'}")

    t0 = time.time()
    for i in range(len(test_ds)):
        lr_tensor, fname = test_ds[i]
        pred = predict_single(model, lr_tensor.unsqueeze(0))

        if CFG.clamp_output:
            pred = np.clip(pred, 0.0, 1.0)

        pred = pred.astype(np.float32)
        assert pred.shape == (256, 256)
        assert not np.isnan(pred).any()
        assert not np.isinf(pred).any()

        np.save(os.path.join(SUB_DIR, fname), pred)
        if (i+1) % 50 == 0 or (i+1) == len(test_ds):
            print(f"  {i+1}/{len(test_ds)}  ({(i+1)/(time.time()-t0):.1f} img/s)")

    print(f"  Done in {time.time()-t0:.1f}s")
    return len(test_ds)

In [30]:

def create_submission():
    print("\n" + "=" * 60)
    print("  CREATING SUBMISSION")
    print("=" * 60)

    rows = []
    files = sorted([f for f in os.listdir(SUB_DIR) if f.endswith(".npy")])

    for idx, file in enumerate(files, start=1):
        arr = np.load(os.path.join(SUB_DIR, file))
        assert arr.shape == (256, 256) and arr.dtype == np.float32

        buffer = BytesIO()
        np.save(buffer, arr)
        encoded = base64.b64encode(buffer.getvalue()).decode()
        rows.append({"id": idx, "npy_base64": encoded})

    df = pd.DataFrame(rows)
    df.to_csv("/kaggle/working/submission.csv", index=False)

    print(f"  {len(df)} files encoded")
    print(f"  CSV: /kaggle/working/submission.csv ({os.path.getsize('/kaggle/working/submission.csv')/1e6:.1f} MB)")
    print(df.head())
    return df

In [31]:

model = train_model()
n_pred = generate_predictions(model)
df = create_submission()

print("\n" + "=" * 60)
print("  ✓ SUBMISSION COMPLETE")
print("=" * 60)
print(f"  {n_pred} predictions → {SUB_DIR}/")
print(f"  submission.csv → /kaggle/working/submission.csv")
print("=" * 60)

  TRAINING — ADVANCED PIPELINE
  Train: 2280  |  Val: 120
  Model params: 3,093,144

  Phase 1: 100 epochs, lr=0.0002
  P1 Ep   1/100  loss=0.60741  psnr=21.60  lr=2.0e-04  [0.3m] ★
  P1 Ep   2/100  loss=0.49670  psnr=22.03  lr=2.0e-04  [0.5m] ★
  P1 Ep   3/100  loss=0.47690  psnr=22.40  lr=2.0e-04  [0.8m] ★
  P1 Ep   4/100  loss=0.46501  psnr=22.75  lr=2.0e-04  [1.1m] ★
  P1 Ep   5/100  loss=0.45828  psnr=23.12  lr=2.0e-04  [1.3m] ★
  P1 Ep   6/100  loss=0.45178  psnr=23.55  lr=2.0e-04  [1.6m] ★
  P1 Ep   7/100  loss=0.44828  psnr=24.05  lr=2.0e-04  [1.9m] ★
  P1 Ep   8/100  loss=0.44354  psnr=24.64  lr=2.0e-04  [2.1m] ★
  P1 Ep   9/100  loss=0.43952  psnr=25.17  lr=2.0e-04  [2.4m] ★
  P1 Ep  10/100  loss=0.43509  psnr=25.60  lr=2.0e-04  [2.6m] ★
  P1 Ep  11/100  loss=0.43294  psnr=25.96  lr=1.9e-04  [2.9m] ★
  P1 Ep  12/100  loss=0.42976  psnr=26.27  lr=1.9e-04  [3.2m] ★
  P1 Ep  13/100  loss=0.42875  psnr=26.53  lr=1.9e-04  [3.4m] ★
  P1 Ep  14/100  loss=0.42746  psnr=26.75  lr=1.9e

In [1]:
import os

# To list files in the current working directory
print(os.listdir()) #

# To list files in a specific directory (e.g., the input directory for datasets)
for dirname, _, filenames in os.walk('/kaggle/working/'):
    for filename in filenames:
        print(os.path.join(dirname, filename)) #


['submission', 'checkpoint_ep20.pth', '.virtual_documents', 'best_model.pth', 'submission.csv', 'checkpoint_ep40.pth', 'state.db']
/kaggle/working/checkpoint_ep20.pth
/kaggle/working/best_model.pth
/kaggle/working/submission.csv
/kaggle/working/checkpoint_ep40.pth
/kaggle/working/state.db
/kaggle/working/submission/000198.npy
/kaggle/working/submission/000145.npy
/kaggle/working/submission/000118.npy
/kaggle/working/submission/000033.npy
/kaggle/working/submission/000120.npy
/kaggle/working/submission/000064.npy
/kaggle/working/submission/000186.npy
/kaggle/working/submission/000142.npy
/kaggle/working/submission/000122.npy
/kaggle/working/submission/000075.npy
/kaggle/working/submission/000043.npy
/kaggle/working/submission/000049.npy
/kaggle/working/submission/000011.npy
/kaggle/working/submission/000005.npy
/kaggle/working/submission/000193.npy
/kaggle/working/submission/000063.npy
/kaggle/working/submission/000180.npy
/kaggle/working/submission/000050.npy
/kaggle/working/submission